In [1]:
api_key = "pcsk_3fnjwy_KTXR6EX195JvaqRpbPzMU9Xgr68au4YXdjKSLSrQ71WUEpNd8pnNz7wz3qE3xSN"

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
from pinecone import Pinecone, ServerlessSpec

index_name = "hybrid-search-langchain-pinecone"

pc = Pinecone(api_key= api_key)    ## initialize the Pinecone client

if index_name not in pc.list_indexes().names():

    pc.create_index(
        name = index_name,
        dimension= 3072,        # dimensionality of dense model  
        metric= "dotproduct",    # sparse values supported only for dotproduct
        spec= ServerlessSpec(cloud="aws", region="us-east-1")
    )

In [3]:
index = pc.Index(index_name)
index

Index(host='https://hybrid-search-langchain-pinecone-bnvppws.svc.aped-4627-b74a.pinecone.io')

In [4]:
from langchain_openai import OpenAIEmbeddings
embedding = OpenAIEmbeddings(model = "text-embedding-3-large")

/Users/rachin/Desktop/GenAI/genai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from pinecone_text.sparse import BM25Encoder

bm25_encoder = BM25Encoder().default()
bm25_encoder

In [6]:
sentences=[
    "In 2023, I visited Paris",
        "In 2022, I visited New York",
        "In 2021, I visited New Orleans",

]

In [7]:
bm25_encoder.fit(sentences)
bm25_encoder.dump("bm25_encoder.json")

100%|██████████| 3/3 [00:00<00:00, 54.42it/s]


In [8]:
bm25_encoder = BM25Encoder().load("bm25_encoder.json")

In [9]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

retriever = PineconeHybridSearchRetriever(embeddings= embedding, sparse_encoder= bm25_encoder, index = index)

In [10]:
retriever.add_texts(sentences)

100%|██████████| 1/1 [00:04<00:00,  4.41s/it]


In [14]:
retriever.invoke("which year I  visited paris")

[Document(metadata={'score': 0.52169776}, page_content='In 2023, I visited Paris'),
 Document(metadata={'score': 0.274594903}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.273624063}, page_content='In 2021, I visited New Orleans')]